# 02 ^ 建模:季节性 LME 竞技(RHR/HRV) 互联网档案馆的存檔，存档日期2014-12-02.

适合六级，BIC - 选择线性混合效应模型竞赛在清洗上
公共投入计划中的月组(`dummy_data[_hrv].csv`)——见
[`cohort_io.py`](../src/cohort_io.py) 并写每个模型的动画
`results/{run_tag}/`.在`CONFIG`中每公尺运行一次(`METRIC`); 模型衍生
然后，表格和数字从这些文件中重新生成，由
[`03_model_results.ipynb`](03_model_results.ipynb) .

| 产出 | 文件(在`results/{run_tag}/`之下). |
| :--- | :--- |
| 运行配置/ 稀释记录 | `run_config.json` , `thinning_summary.json` , `thinning_dropped_user_ids.csv` |
| 按模式审计(协同，共变，VIF,验证CSV/SVGs,随机效果). | `<timestamp>_<formula>/…` 包括 `model-row.csv` (中文(简体) ). |
| 所有候选行(每个装配型号1个) | `lme-models-results-{metric}.csv` |
| 总得分(REML) | `model_{metric}_winner.pkl` , `model_{metric}_coefficients_interpreted.csv` |
| OLS 温度剩余参数 | `ols_temperature_params.csv` |
| 阶段检查站(恢复支助) | `checkpoint_state.json` |

** 方法。** 阶段 1 基线 (年龄×性别) → 2 季节性(谐振对日长度)
□ 3季节性×日间相互作用(诊断:装配和上报，未携带
∗∗∗∗∗∗∗ 4 空间(纬度±经度，±季节相互作用)
− 5 温度(平均对最大/最小，±季节相互作用) − 6 与
temperature **residualised** on latitude/season (OLS) to break collinearity.
获胜者由BIC(ML适配)选择;整体获胜者与REML重排，用于报告。
进步生活[`stage_competition.py`](../src/stage_competition.py)(单位测试)
`STAGE_OVERRIDES`中，`CONFIG`迫使不同的候选人进步。
(例如`{'Stage1': '1.1_Linear'}`)用于反事实路径;结果CSV然后记录
`bic_winner`/`progressed`/`forced`/`on_path`每行。

快速随机拦截适配器生活在[`lme_binned.py`](../src/lme_binned.py);
[`seasonality_lib.py`](../src/seasonality_lib.py) 中的建模助手。文件访问将启动
)通过[`file_io.py`](../src/file_io.py)(`./data`或`$WPM_DATA_DIR`.

** 续集。**每个完成的模型都在其文件夹中写 `model-row.csv` , 每一个
完成阶段更新 `checkpoint_state.json`; 重运行同样的笔记本
`CONFIG` 跳过已完成的工作。运行标记绑在一个数据配置上:笔记本
当`run_config.json`(测量、稀释、亚样本分量、种子)时，拒绝继续
或检查点(目标、候选规格、覆盖、样本大小)
设置 `FORCE_CLEAN = True` 重新开始。

UN - 区域和国家趋势地块至少需要每个组的500用户
`MIN_USERS_PER_GROUP`,在`CONFIG`中;为更小的组别降低。

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import gc
import io
import json
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(
    0, os.path.abspath('../src')
)  # 图书馆模块(公开发布)
import cohort_io as cio
import file_io as fio
import lme_binned
import paper_tables
import seasonality_lib as sl
import stage_competition

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
METRIC = 'rhr'  # 'rhr' | 'hrv'
THIN_RHR = True  # 基于密度的年龄稀释 -- 22 人工活性( A) RHR 仅限)
RHR_RUN_TAG = 'rhr_rerun'  # RHR 运行的结果/ 子文件夹
HRV_RUN_TAG = 'hrv_rerun'  # HRV 运行的结果/ 子文件夹
SUBSAMPLE_FRAC = 0.10  # 用于装配的用户比例(百分比) 1.0 =每个人).
MAX_STAGE = 6  # 在此阶段后停止( 1-6 )
# 反事实进展:强迫候选人(或互动决定)
# 无论BIC,如{'Stage1':`1。1_Linear}或
# {' Stage4_Interaction ': False].  ⁇ =纯 BIC 选择。
# 决定点:第1阶段，第2阶段，Stage4_Base,Stage4_Interaction,
# Stage5_Base , Stage5_Interaction ,第6阶段(见 stage_competition .py; 阶段 3
# 只有诊断，没有决定点。
STAGE_OVERRIDES = {}
GENERATE_PLOTS = False  # *每个候选人的完整诊断图集(慢)
FORCE_CLEAN = False  # 忽略现有的检查站/ 已完成的模型
RANDOM_STATE = 42
# UN 区域/不同用户较少的国家被排除在外。
# 趋势地块； 降低小群。
MIN_USERS_PER_GROUP = 500

METRIC_CFG = {
    'rhr': dict(
        colname='avg_resting_heart_rate_bpm',
        units='bpm',
        title='Resting Heart Rate',
        summary_range=(30.0, 150.0),
    ),
    'hrv': dict(
        colname='avg_hrv',
        units='ms',
        title='Heart Rate Variability',
        summary_range=(5.0, 250.0),
    ),
}
cfg = METRIC_CFG[METRIC]
colname, units, colname_title = cfg['colname'], cfg['units'], cfg['title']
# HRV是仿照日志比例(系数为%变化)的模型。
colname_c = 'target_log' if METRIC == 'hrv' else colname
thinned = THIN_RHR and METRIC == 'rhr'
run_tag = RHR_RUN_TAG if METRIC == 'rhr' else HRV_RUN_TAG
STAGE_OVERRIDES = stage_competition.validate_overrides(STAGE_OVERRIDES)
if STAGE_OVERRIDES:
  print(
      f'STAGE_OVERRIDES active: {STAGE_OVERRIDES} -> use a dedicated run tag!'
  )

data_dir = fio.get_data_dir()
extra_dir = fio.get_extra_dir(data_dir)
results_dir = fio.get_results_dir(run_tag, data_dir)
fio.make_dirs(results_dir)
run_config = dict(
    metric=METRIC,
    thin_rhr=thinned,
    subsample_frac=SUBSAMPLE_FRAC,
    max_stage=MAX_STAGE,
    stage_overrides=STAGE_OVERRIDES,
    random_state=RANDOM_STATE,
    started=time.strftime('%Y-%m-%d %H:%M:%S'),
)
# 运行标记属于一个数据配置:拒绝恢复(或重复使用)
# 当定义匹配样本的设置不同时。
run_config_path = os.path.join(results_dir, 'run_config.json')
if fio.file_exists(run_config_path) and not FORCE_CLEAN:
  with fio.open_file(run_config_path) as f:
    previous = json.load(f)
  data_keys = ('metric', 'thin_rhr', 'subsample_frac', 'random_state')
  changed = {
      k: (previous.get(k), run_config[k])
      for k in data_keys
      if previous.get(k) != run_config[k]
  }
  if changed:
    raise ValueError(
        f'results/{run_tag} was produced with different data settings'
        f' {changed} (previous, current); set FORCE_CLEAN = True or use'
        ' another run tag.'
    )
with fio.open_file(run_config_path, 'w') as f:
  json.dump(run_config, f, indent=2)
print(
    f'{METRIC.upper()}  thinned={thinned}  frac={SUBSAMPLE_FRAC}  ->'
    f' {results_dir}'
)

In [ ]:
NEEDED_COLS = [
    'user_id',
    'activity_period',
    'gender',
    'age',
    'lat_e7',
    'lng_e7',
    colname,
    'un_region',
    'country_2_alpha_cd',
    'dl_monthly',
    'avg_temp_mean',
    'avg_temp_min',
    'avg_temp_max',
]

t0 = time.time()
# cohort_io将公众CSV标题和单位映射到列名上，列名被使用。
# 密码
input_path = cio.resolve_input_path(data_dir, METRIC)
frhr = cio.load_cohort(data_dir, METRIC, columns=NEEDED_COLS, path=input_path)
print(
    f'Loaded {len(frhr):,} user-months / {frhr["user_id"].nunique():,} users'
    f' from {input_path} in {time.time() - t0:.0f}s'
)

In [ ]:
# ----------------------------------------------------------------------------
# 年龄的瘦化——22 假出生年动脉
# 必须发生 BEFORE 子样本和建模。组表/ 数字
# 描述稀薄组群由01_cohort_stats.ipynb生产。
# ----------------------------------------------------------------------------
if thinned:
  frhr, thin_stats = sl.apply_density_based_thinning(
      frhr,
      target_col=colname,
      val_min=cfg['summary_range'][0],
      val_max=cfg['summary_range'][1],
      random_state=RANDOM_STATE,
  )
  with fio.open_file(
      os.path.join(results_dir, 'thinning_dropped_user_ids.csv'), 'w'
  ) as f:
    pd.Series(sorted(thin_stats['dropped_user_ids']), name='user_id').to_csv(
        f, index=False
    )
  with fio.open_file(
      os.path.join(results_dir, 'thinning_summary.json'), 'w'
  ) as f:
    json.dump(
        {k: v for k, v in thin_stats.items() if isinstance(v, (int, float))},
        f,
        indent=2,
    )
  print(
      f'After thinning: {len(frhr):,} user-months /'
      f' {frhr["user_id"].nunique():,} users'
  )

In [ ]:
# ----------------------------------------------------------------------------
# 地物工程(共享执行：seasonality_lib),也用于
# publication_figures. py so 数字和模型见相同的预测器).
# ----------------------------------------------------------------------------
frhr['age_group'] = pd.cut(
    frhr['age'], bins=np.arange(18, 90, 10), include_lowest=True
)
frhr['lat_group'] = pd.cut(
    frhr['lat_e7'] / 1e7, bins=[-60, -30, 0, 30, 60, 90], include_lowest=True
)

# 性别： 中二进制( 男性 = 1 所以拦截是人口之意；
# ⁇ 纬度 ⁇ ;圆上经度。
male_share = sl.add_sex_and_longitude_features(frhr)

# 连续预测器： Gelman 标准化(中心，除以 2SD)
# 以FULL组数计算(在子采样之前)。
# 统计保存到逆变系数并写入
# (feature_stats.json,以下),这样03_model_results.ipynb重建了该建筑。
# 预测器完全匹配。
feature_summary = sl.standardise_model_features(frhr)
scaling_factors = sl.scaling_factors_from_summary(feature_summary)

# 季节性口琴(年度+半年).
sl.add_month_harmonics(frhr)

if METRIC == 'hrv':
  frhr['target_log'] = np.log(frhr[colname])
optimal_lambda = (
    0 if METRIC == 'hrv' else None
)  # sl 绘图的日志转换旗

## LME 模式竞争

In [ ]:
# ----------------------------------------------------------------------------
# 安装用户子样本( 用户为 QQ) 2 月),紧凑型号
# ----------------------------------------------------------------------------
frhr['user_id_n'] = frhr['user_id'].astype('category').cat.codes
unique_users = frhr['user_id_n'].unique()
if SUBSAMPLE_FRAC < 1.0:
  sampled = pd.Series(unique_users).sample(
      frac=SUBSAMPLE_FRAC, random_state=RANDOM_STATE
  )
  frhr_subsample = frhr[frhr['user_id_n'].isin(sampled)].copy()
else:
  frhr_subsample = frhr
del frhr  # 不需要全帧(从负载单元格中重跑以取回)
gc.collect()

user_counts = frhr_subsample['user_id_n'].value_counts()
frhr_subsample = frhr_subsample[
    frhr_subsample['user_id_n'].isin(user_counts[user_counts > 1].index)
].copy()
user_mapping_df = frhr_subsample[['user_id_n', 'user_id']].drop_duplicates()
frhr_subsample = frhr_subsample.drop(columns=['user_id'])
float_cols = frhr_subsample.select_dtypes(include=['float64']).columns
frhr_subsample[float_cols] = frhr_subsample[float_cols].astype('float32')
print(
    f'Fitting sample: {frhr_subsample["user_id_n"].nunique():,} users,'
    f' {len(frhr_subsample):,} rows'
)

# ----------------------------------------------------------------------------
# 温度在纬度x季(正交化，阶段6)
# 与 sl. TEMP_RESIDUAL_FORMULA中的OLS;其参数为：
# 表格标签：{度量} _ols_params。
# ----------------------------------------------------------------------------
ols_info = sl.residualise_temperature(frhr_subsample, method='harmonic')
for temp_var, info in ols_info.items():
  frhr_subsample[f'{temp_var}_res'] = frhr_subsample[f'{temp_var}_res'].astype(
      'float32'
  )
  print(f'  {temp_var}_res  (OLS R2 = {info["rsquared"]:.4f})')
for col in ['avg_temp_max_sq_c_res', 'avg_temp_min_sq_c_res']:
  frhr_subsample[col] = frhr_subsample[col].astype('float32')
sl.write_feature_stats(
    results_dir, sl.feature_stats(feature_summary, ols_info, male_share)
)
paper_tables.write_ols_temperature_params(
    paper_tables.fit_ols_temperature_params(frhr_subsample),
    data_dir,
    METRIC,
    results_subdir=run_tag,
)
gc.collect()

# ISO-2 -> ISO-3,用于图案残存地图(extra_data/unsd-countries.csv);该地图为：
# 引用表不可用时跳过(未使用 extra_data 的公开取出)。
_unsd_path = os.path.join(extra_dir, 'unsd-countries.csv')
if fio.file_exists(_unsd_path):
  with fio.open_file(_unsd_path, 'r') as f:
    # keep_default_na=False: Namibia's code is 'NA'.
    ISO2_TO_ISO3 = (
        pd.read_csv(f, keep_default_na=False, na_values=[''], dtype=str)
        .set_index('ISO-alpha2 Code')['ISO-alpha3 Code']
        .to_dict()
    )
  # 编码 UNSD 表格缺少但地图绘图(台湾); 同一补充
  # 作为 publication_figures . ISO2_TO_ISO3_SUPPLEMENT .
  ISO2_TO_ISO3.update({'TW': 'TWN'})
else:
  ISO2_TO_ISO3 = None
  print(f'{_unsd_path} not found: country residual maps will be skipped')

In [ ]:
# ----------------------------------------------------------------------------
# 每个阶段的候选右手边都位于stage_competition. DEFAULT_SPEC.
# (年龄形态，口琴，纬度形态，温度形态);在这里绘制组图。
# ----------------------------------------------------------------------------
SPEC = stage_competition.DEFAULT_SPEC
print('Stage 1 candidates:', dict(SPEC.stage1))
PLOT_GROUPS = [
    None,
    'un_region',
    'gender',
    'lat_group',
    'age_group',
    'country_2_alpha_cd',
]
PLOT_SUFFIX = {
    None: '',
    'un_region': '_un_region',
    'gender': '_gender',
    'lat_group': '_lat_bin',
    'age_group': '_age_group',
    'country_2_alpha_cd': '_country',
}
VALIDATION = [  # (组 col,文件干，标题，行密钥，排除级别)
    (
        'country_2_alpha_cd',
        'country',
        'Country-Level',
        'aggregate_R_country',
        None,
    ),
    (
        'un_region',
        'un_region',
        'UN-Region-Level',
        'aggregate_R_un_region',
        'oceania',
    ),
    ('gender', 'gender', 'Gender-Level', 'aggregate_R_gender', 'other'),
    ('lat_group', 'lat_bin', 'Latitude-Bin-Level', 'aggregate_R_lat_bin', None),
    ('age_group', 'age_group', 'Age-Level', 'aggregate_R_age_group', None),
]

In [ ]:
# ----------------------------------------------------------------------------
# 一个型号适合 =一个呼叫。可续用:一个有 model-row.csv 的文件夹被重用
# (以上CONFIG的检查是同一份数据上安装的。)
# ----------------------------------------------------------------------------
def _read_csv(path, **kwargs):
  with fio.open_file(path, 'r') as f:
    return pd.read_csv(f, **kwargs)


def _find_completed(formula):
  tag = formula.replace(' ', '')
  for sub in fio.list_dir(results_dir):
    row_file = os.path.join(results_dir, sub, 'model-row.csv')
    if not fio.file_exists(row_file):
      continue
    row = _read_csv(row_file)
    if not row.empty and str(row['formula'].iloc[0]).replace(' ', '') == tag:
      vif_file = os.path.join(results_dir, sub, 'vif.csv')
      vif = _read_csv(vif_file) if fio.file_exists(vif_file) else None
      return row.iloc[0].to_dict(), vif
  return None, None


def _save_fig(fig, path):
  with fio.open_file(path, 'w') as f:
    fig.savefig(f, format='svg', dpi=300)
  plt.close(fig)


def fit_lme_model(
    formula,
    name,
    generate_plots=False,
    only_latitude_plots=False,
    skip_hessian=False,
    save_random_effects=False,
):
  """Fits one candidate, writes its audit folder, returns (result, model_row, vif)."""
  print(f'\n{"=" * 70}\n{name}: {formula}\n{"=" * 70}')
  if not FORCE_CLEAN:
    row, vif = _find_completed(formula)
    if row is not None:
      print('  [model checkpoint] already fitted - skipping')
      return None, row, vif

  output_path = sl.get_create_output_path(formula, results_dir, colname, METRIC)
  model = lme_binned.BinnedMixedLM.from_formula(
      formula, data=frhr_subsample, groups='user_id_n'
  )
  result = model.fit(reml=False, method='lbfgs', skip_hessian=skip_hessian)
  blups = lme_binned.inject_fast_random_effects(result)
  frhr_subsample['fitted_values'] = result.fittedvalues
  frhr_subsample['predicted_val'] = result.predict()

  model_row, vif = sl.extract_model_stats(
      result,
      model_name=name,
      formula=formula,
      data=frhr_subsample,
      target_col=colname_c,
      scaling_factors=scaling_factors,
  )

  with fio.open_file(output_path + 'vif.csv', 'w') as f:
    vif.to_csv(f, index=False)
  ci = result.conf_int()
  coef_df = pd.DataFrame({
      'Coef.': result.params,
      'Std.Err.': result.bse,
      'z': result.tvalues,
      'P>|z|': result.pvalues,
      '[0.025': ci[0],
      '0.975]': ci[1],
  })
  with fio.open_file(output_path + 'model_coefficients.csv', 'w') as f:
    coef_df.to_csv(f, index=True)
  cov_df = pd.DataFrame(
      result.cov_params(), index=result.param_names, columns=result.param_names
  )
  with fio.open_file(output_path + 'cov_params.csv', 'w') as f:
    cov_df.to_csv(f, index=True)
  if save_random_effects:
    re_df = pd.DataFrame(
        {'user_id_n': blups.index, 'random_intercept': blups.values}
    ).merge(user_mapping_df, on='user_id_n')[['user_id', 'random_intercept']]
    with fio.open_file(output_path + 'random_effects.parquet', 'wb') as f:
      re_df.to_parquet(f, engine='pyarrow', compression='snappy')

  if generate_plots:
    fig_re, fig_resid = sl.summary_report(result, frhr_subsample, colname_title)
    _save_fig(fig_re, output_path + 'random_effects.svg')
    _save_fig(fig_resid, output_path + 'residuals.svg')
  else:
    sl.compute_r2(result, frhr_subsample)

  if generate_plots or only_latitude_plots:
    # 5 %的用户(QQ 10k)对于趋势图来说是足够的。
    n_plot = max(10_000, int(0.05 * frhr_subsample['user_id_n'].nunique()))
    rng = np.random.RandomState(RANDOM_STATE)
    plot_users = rng.choice(
        frhr_subsample['user_id_n'].unique(),
        size=min(n_plot, frhr_subsample['user_id_n'].nunique()),
        replace=False,
    )
    plot_df = frhr_subsample[
        frhr_subsample['user_id_n'].isin(plot_users)
    ].copy()

    groups = ['lat_group'] if only_latitude_plots else PLOT_GROUPS
    modes = [False] if only_latitude_plots else [True, False]
    for group_col in groups:
      for use_re in modes:
        stem = 'prediction_fit' if use_re else 'calibration_fit'
        fig = sl.plot_seasonal_trends(
            plot_df,
            result,
            col=colname_c,
            col_name=colname_title,
            group_col=group_col,
            use_random_effects=use_re,
            boxcox_lambda=optimal_lambda,
            min_users_per_group=MIN_USERS_PER_GROUP,
        )
        _save_fig(fig, f'{output_path}{stem}{PLOT_SUFFIX[group_col]}.svg')

    validations = (
        [v for v in VALIDATION if v[0] == 'lat_group']
        if only_latitude_plots
        else VALIDATION
    )
    for col, stem, title, row_key, exclude in validations:
      v_df = (
          frhr_subsample
          if exclude is None
          else frhr_subsample[frhr_subsample[col] != exclude]
      )
      v_fig, r_val, slope_val, agg_df = sl.save_aggregate_validation(
          v_df,
          result,
          group_col=col,
          target_col=colname_c,
          title_prefix=title,
          boxcox_lambda=optimal_lambda,
      )
      if v_fig is None:
        continue
      model_row[row_key] = r_val
      model_row[f'calib_slope_{col}'] = slope_val
      if col == 'un_region':
        bias = agg_df['actual_mean'] - agg_df['predicted_mean']
        model_row.update(
            max_regional_bias=bias.max(),
            min_regional_bias=bias.min(),
            peak_regional_uncertainty=bias.abs().max(),
            regional_bias_std=bias.std(),
        )
      _save_fig(v_fig, f'{output_path}aggregate_validation_{stem}.svg')
      with fio.open_file(f'{output_path}validation_data_{stem}.csv', 'w') as f:
        agg_df.to_csv(f, index=False)
      if col == 'country_2_alpha_cd' and ISO2_TO_ISO3 is not None:
        agg_df['country_3_alpha_cd'] = agg_df['country_2_alpha_cd'].map(
            ISO2_TO_ISO3
        )
        unmapped = agg_df['country_3_alpha_cd'].isna()
        if unmapped.any():
          codes = sorted(agg_df.loc[unmapped, 'country_2_alpha_cd'].unique())
          print(f'  [maps] dropped unmapped ISO-2 code(s): {codes}')
          agg_df = agg_df[~unmapped]
        fig = sl.plot_global_residuals(
            agg_df,
            iso_col='country_3_alpha_cd',
            obs_col='actual_mean',
            pred_col='predicted_mean',
            is_hrv=(METRIC == 'hrv'),
        )
        with fio.open_file(
            f'{output_path}global_residuals_{METRIC}.html', 'w'
        ) as f:
          f.write(fig.to_html(include_plotlyjs='cdn'))

  with fio.open_file(output_path + 'model-row.csv', 'w') as f:
    pd.DataFrame([model_row]).to_csv(f, index=False)

  plt.close('all')
  frhr_subsample.drop(
      columns=['fitted_values', 'predicted_val'], inplace=True, errors='ignore'
  )
  gc.collect()
  return result, model_row, vif

In [ ]:
# ----------------------------------------------------------------------------
# 阶段驱动(stage_competition. run_competition) 设有检查站： 简历。
# 从最后一个完成的阶段；STAGE_OVERRIDES 强制反事实路径。
# ----------------------------------------------------------------------------
checkpoint_path = os.path.join(results_dir, 'checkpoint_state.json')
if FORCE_CLEAN:
  fio.remove_file(checkpoint_path)
state = lme_binned.load_checkpoint(
    checkpoint_path
) or stage_competition.new_state(STAGE_OVERRIDES)

state, final_df = stage_competition.run_competition(
    fit_lme_model,
    colname_c,
    state,
    spec=SPEC,
    max_stage=MAX_STAGE,
    overrides=STAGE_OVERRIDES,
    checkpoint_fn=lambda s: lme_binned.save_checkpoint(checkpoint_path, s),
    fit_kwargs=dict(generate_plots=GENERATE_PLOTS),
    # 存放在检查站:用不同样本书写的检查站是：
    # 未恢复。
    data_key=dict(
        n_rows=len(frhr_subsample),
        n_users=int(frhr_subsample['user_id_n'].nunique()),
        subsample_frac=SUBSAMPLE_FRAC,
        thinned=thinned,
    ),
)

results_csv = os.path.join(results_dir, f'lme-models-results-{METRIC}.csv')
with fio.open_file(results_csv, 'w') as f:
  final_df.to_csv(f, index=False)
print(f'\nSaved {len(final_df)} candidate rows to {results_csv}')
display(
    final_df[[
        'stage',
        'candidate',
        'BIC',
        'bic_winner',
        'progressed',
        'forced',
        'on_path',
    ]]
)

In [ ]:
# ----------------------------------------------------------------------------
# 在所有路径列上的总赢家(最低BIC;无STAGE_OVERRIDES)
# 每行都在路上：REML 重修，保存，最后的地块
# ----------------------------------------------------------------------------
best = stage_competition.overall_winner(final_df)
winner_name, winner_formula = best['model'], best['formula']
print(
    f'OVERALL WINNER: {winner_name}\n  {winner_formula}\n  BIC ='
    f' {best["BIC"]:.2f}'
)

# BIC选择使用ML(比较固定效果需要);
# 模型重排为 REML,用于无偏差组件 / SEs。
reml_model = lme_binned.BinnedMixedLM.from_formula(
    winner_formula, data=frhr_subsample, groups='user_id_n'
)
reml_res = reml_model.fit(reml=True, method='lbfgs')
lme_binned.inject_fast_random_effects(reml_res)

ci = reml_res.conf_int()
coeff_df = pd.DataFrame({
    'coef': reml_res.params,
    'std_err': reml_res.bse,
    'z': reml_res.tvalues,
    'P_z': reml_res.pvalues,
    'ci_lower': ci[0],
    'ci_upper': ci[1],
})
if METRIC == 'hrv':  # 日志比例 -> 百分比变化
  for src, dst in [
      ('coef', 'percent_change'),
      ('ci_lower', 'percent_ci_lower'),
      ('ci_upper', 'percent_ci_upper'),
  ]:
    coeff_df[dst] = (np.exp(coeff_df[src]) - 1) * 100
with fio.open_file(
    os.path.join(results_dir, f'model_{METRIC}_coefficients_interpreted.csv'),
    'w',
) as f:
  coeff_df.to_csv(f, index=True)
display(coeff_df)

# 腌制REML结果，没有设计矩阵。
reml_res._cache.clear()
for attr in ['endog', 'exog', 'groups', 'exog_re', 'binned_data']:
  setattr(reml_res.model, attr, None)
reml_res.model.data.frame = reml_res.model.data.frame.iloc[:1]
for attr in ['orig_endog', 'orig_exog', 'endog', 'exog', 'groups']:
  setattr(reml_res.model.data, attr, None)
buf = io.BytesIO()
reml_res.save(buf)
with fio.open_file(
    os.path.join(results_dir, f'model_{METRIC}_winner.pkl'), 'wb'
) as f:
  f.write(buf.getvalue())

# 最终诊断+校准胜者图案(全集),随机效果保存。
FORCE_CLEAN = True  # 绕过模式检查点，重新生成地块
try:
  _, _, winner_vif = fit_lme_model(
      winner_formula,
      f'Final_Winner_{winner_name}',
      generate_plots=True,
      save_random_effects=True,
  )
finally:
  FORCE_CLEAN = False
high = winner_vif[winner_vif['GVIF_Adj'] > 2.0]
print(
    'Collinearity audit:',
    'PASS (all adjusted GVIF <= 2)' if high.empty else f'\n{high}',
)

In [ ]:
# 哪些候选人在每条衡量标准上最好(BIC 选择的系统检查)。
cols = {
    'BIC': 'min',
    'AIC': 'min',
    'marginal_R2': 'max',
    'conditional_R2': 'max',
    'rmse': 'min',
    'aggregate_R_lat_bin': 'max',
    'aggregate_R_country': 'max',
}
for col, op in cols.items():
  if col in final_df and final_df[col].notna().any():
    idx = final_df[col].idxmin() if op == 'min' else final_df[col].idxmax()
    print(f'{op} {col:20s}: {final_df.loc[idx, "model"]}')

** 下一步：** 将本笔记本运行到另一个度量，然后重生模型生成的
表格和数字与[`03_model_results.ipynb`](03_model_results.ipynb),指
`RUN_TAGS` 时间 `RHR_RUN_TAG` / `HRV_RUN_TAG` .